In [1]:
import torch
from icecream import ic

In [2]:
device = torch.device("cpu")

In [3]:
positions = torch.rand(100, 100_000, 2).to(device)
histogram_shape = (500, 500)
num_hist_dims = 2
clamped_corner_positions_in_bin_space = (
    torch.rand(100, 100_000, 4, num_hist_dims).to(device).long()
)
corner_charges = torch.rand(100, 100_000, 4).to(device)

In [4]:
%%timeit
vector_shape = positions.shape[:-2]
charge_grid = corner_charges.new_zeros(*vector_shape, *histogram_shape)

vector_indices = tuple(
    torch.arange(this_vector_dim_length).reshape(
        this_vector_dim_length, *([1] * (num_hist_dims - dim_idx + 1))
    )
    for dim_idx, this_vector_dim_length in enumerate(vector_shape)
)

charge_grid = charge_grid.index_put(
    vector_indices + clamped_corner_positions_in_bin_space.unbind(-1),
    corner_charges,
    accumulate=True,
)

9.33 s ± 540 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
del (
    positions,
    histogram_shape,
    num_hist_dims,
    clamped_corner_positions_in_bin_space,
    corner_charges,
)

In [6]:
first_pos = torch.rand(100, 100_000).to(device)
num_dims = 2
weights = torch.ones(100, 100_000).to(device)
bins = (
    torch.linspace(0.0, 1.0, 500).to(device),
    torch.linspace(0.0, 1.0, 500).to(device),
)
positions = (first_pos, torch.rand(100, 100_000).to(device))
grid_sizes = []
spacings = []
for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)
# Normalize particle coordinates to grid index space
grid_indices = []
fractional_parts = []
for pos, bin_array, spacing in zip(positions, bins, spacings):
    # Normalized coordinate in grid index space
    u = (pos - bin_array[0]) / spacing

    # Left cell index
    i = torch.floor(u).to(torch.int64)
    grid_indices.append(i)

    # Fractional distance to right cell
    w = u - i
    fractional_parts.append(w)
# Generate all corner combinations and their weights
num_corners = 2**num_dims
corner_indices = []
corner_weights = []
for corner in range(num_corners):
    # Determine which corners to use based on binary representation
    corner_offsets = []
    weight_factors = []

    for dim in range(num_dims):
        if (corner >> dim) & 1:  # Use right cell in this dimension
            corner_offsets.append(1)
            weight_factors.append(fractional_parts[dim])
        else:  # Use left cell in this dimension
            corner_offsets.append(0)
            weight_factors.append(1 - fractional_parts[dim])

    # Calculate indices for this corner
    corner_idx_list = []
    for dim in range(num_dims):
        base_idx = grid_indices[dim]
        offset_idx = (base_idx + corner_offsets[dim]).clamp(0, grid_sizes[dim] - 1)
        corner_idx_list.append(offset_idx)

    # Calculate weight for this corner
    corner_weight = weights
    for weight_factor in weight_factors:
        corner_weight = corner_weight * weight_factor

    corner_indices.append(corner_idx_list)
    corner_weights.append(corner_weight)

In [7]:
%%timeit
# Convert multi-dimensional indices to flat indices
def multi_to_flat_index(idx_list):
    flat_idx = idx_list[0]
    stride = 1
    for dim in range(1, num_dims):
        stride *= grid_sizes[dim - 1]
        flat_idx = flat_idx + idx_list[dim] * stride
    return flat_idx


# Flatten batch dims and particle dim together
batch_shape = first_pos.shape[:-1]
B = int(torch.tensor(batch_shape).prod()) if batch_shape else 1
N = first_pos.shape[-1]


def flatten_tensor(t):
    return t.reshape(B, N)


# Prepare all indices and weights for batch processing
all_flat_indices = []
all_weights = []
for corner_idx_list, corner_weight in zip(corner_indices, corner_weights):
    flat_idx = multi_to_flat_index(corner_idx_list)
    all_flat_indices.append(flatten_tensor(flat_idx))
    all_weights.append(flatten_tensor(corner_weight))

# Concatenate all indices and weights
idx_all = torch.cat(all_flat_indices, dim=1)  # shape (B, num_corners * N)
vals_all = torch.cat(all_weights, dim=1)  # shape (B, num_corners * N)

# Output buffer
total_grid_size = int(torch.tensor(grid_sizes).prod())
charge = torch.zeros((B, total_grid_size), dtype=vals_all.dtype, device=vals_all.device)

# Vectorized batched index_add_
for b in range(B):
    charge[b].index_add_(0, idx_all[b], vals_all[b])

# Compute inverse cell volume
# cell_volume = 1.0
# for spacing in spacings:
#     cell_volume *= spacing
# inv_cell_volume = 1.0 / cell_volume
# charge = charge * inv_cell_volume

# Reshape back to original batch dimensions + grid dimensions
out_shape = (*batch_shape, *grid_sizes[::-1])
charge = charge.reshape(out_shape)  # Grid dimensions are reversed by the reshape

batch_ndim = len(batch_shape)
spatial_axes = list(range(batch_ndim, batch_ndim + num_dims))

# Permute to put spatial axes in the correct order
final_result_to_return = charge.permute(*range(batch_ndim), *reversed(spatial_axes))

207 ms ± 13.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
del (
    first_pos,
    num_dims,
    weights,
    bins,
    positions,
    grid_sizes,
    spacings,
    N,
    spacing,
    grid_indices,
    fractional_parts,
    u,
    i,
    w,
    num_corners,
    corner_indices,
    corner_weights,
    corner_offsets,
    weight_factors,
    corner_idx_list,
    base_idx,
    offset_idx,
    corner_weight,
)

In [9]:
positions = torch.rand(100, 100_000, 2).to(device)
histogram_shape = (500, 500)
num_hist_dims = 2
clamped_corner_positions_in_bin_space = (
    torch.rand(100, 100_000, 4, num_hist_dims).to(device).long()
)
corner_charges = torch.rand(100, 100_000, 4).to(device)

In [10]:
%%timeit
vector_shape = positions.shape[:-2]
charge_grid = corner_charges.new_zeros(*vector_shape, *histogram_shape)

vector_indices = tuple(
    torch.arange(this_vector_dim_length).reshape(
        this_vector_dim_length, *([1] * (num_hist_dims - dim_idx + 1))
    )
    for dim_idx, this_vector_dim_length in enumerate(vector_shape)
)

charge_grid = charge_grid.index_put(
    vector_indices + clamped_corner_positions_in_bin_space.unbind(-1),
    corner_charges,
    accumulate=True,
)

7.95 s ± 527 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
del (
    positions,
    histogram_shape,
    num_hist_dims,
    clamped_corner_positions_in_bin_space,
    corner_charges,
)